<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02_bigquery/03_ENARES_2024_STAGE2_load_spss_metadata_to_bigquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 03_ENARES_2024_STAGE2_load_spss_metadata_to_bigquery.ipynb
# Stage 2 - Load SPSS metadata to BigQuery
# Corrected version: PDF governance + reproducible config
# ============================================================

!pip install -q google-cloud-bigquery pandas pyreadstat pandas-gbq pyarrow google-api-python-client

from google.colab import auth, drive
from google.cloud import bigquery
from googleapiclient.discovery import build
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import os
import re
import hashlib
import pyreadstat

auth.authenticate_user()
drive.mount("/content/drive")

# Prefer environment variable for reproducibility; keep input fallback for Colab.
PROJECT_ID = os.environ.get("PROJECT_ID") or input("Enter your Google Cloud PROJECT_ID: ").strip()
if not PROJECT_ID:
    raise ValueError("PROJECT_ID cannot be empty.")

LOCATION = os.environ.get("BQ_LOCATION", "US")

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
SAV_DIR = f"{ROOT_DRIVE}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"
DOCS_DIR = f"{ROOT_DRIVE}/docs"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# Drive API is used only to resolve Google Drive file IDs for local PDFs when possible.
# If IDs cannot be resolved automatically, the notebook creates a manual registry template.
drive_service = build("drive", "v3")

print("Using project:", PROJECT_ID)
print("Using LOG_DIR:", LOG_DIR)
print("Using DOCS_DIR:", DOCS_DIR)
print("RUN_UTC:", RUN_UTC)

In [ ]:
# ============================================================
# Stage 2 metadata constants
# ============================================================

METADATA_DATASET = "enares2024_crs04_raw"

METADATA_TABLES = [
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
]

In [ ]:
# ============================================================
# 1. Load source file mapping from Notebook 2
# ============================================================

source_check_path = f"{LOG_DIR}/ENARES_2024_STAGE2_source_file_check.csv"

if not os.path.exists(source_check_path):
    raise FileNotFoundError(
        "Missing source file check CSV. Run Notebook 2 first."
    )

table_mapping = pd.read_csv(source_check_path)

required_cols = ["module", "chapter", "source_file", "target_table", "source_path"]

missing_cols = [c for c in required_cols if c not in table_mapping.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in source check CSV: {missing_cols}")

table_mapping["file_exists"] = table_mapping["source_path"].apply(os.path.exists)

if not table_mapping["file_exists"].all():
    print(table_mapping.to_string(index=False))
    raise FileNotFoundError("At least one .sav path from Notebook 2 no longer exists.")

display(table_mapping)

In [ ]:
# ============================================================
# 2. Extract SPSS metadata
# Critical point: use meta.variable_value_labels when available
# so labels stay associated with each variable.
# ============================================================

variables_rows = []
value_label_rows = []
missing_rows = []

for _, row in table_mapping.iterrows():
    print(f"Reading metadata: {row['source_file']}")

    _, meta = pyreadstat.read_sav(
        row["source_path"],
        metadataonly=True,
        apply_value_formats=False,
    )

    chapter = row["chapter"]
    source_file = row["source_file"]
    target_table = row["target_table"]

    # --------------------------------------------------------
    # Variable labels
    # --------------------------------------------------------
    for var_name, var_label in zip(meta.column_names, meta.column_labels):
        variables_rows.append({
            "chapter": chapter,
            "source_file": source_file,
            "target_table": target_table,
            "variable_name": var_name,
            "variable_label": var_label,
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        })

    # --------------------------------------------------------
    # Value labels
    # --------------------------------------------------------
    variable_value_labels = getattr(meta, "variable_value_labels", {}) or {}

    for var_name, labels in variable_value_labels.items():
        for value, label in labels.items():
            value_label_rows.append({
                "chapter": chapter,
                "source_file": source_file,
                "target_table": target_table,
                "variable_name": var_name,
                "value": str(value),
                "label": str(label),
                "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            })

    # --------------------------------------------------------
    # Missing codes (one value/range per row)
    # --------------------------------------------------------
    missing_user_values = getattr(meta, "missing_user_values", {}) or {}
    missing_ranges = getattr(meta, "missing_ranges", {}) or {}

    # User-defined missing values
    for var_name, values in missing_user_values.items():
        if not isinstance(values, (list, tuple, set)):
            values = [values]

        for value in values:
            missing_rows.append({
                "chapter": chapter,
                "source_file": source_file,
                "target_table": target_table,
                "variable_name": var_name,
                "missing_type": "user_value",
                "missing_value": str(value),
                "missing_range_low": None,
                "missing_range_high": None,
                "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            })

    # Missing ranges
    for var_name, ranges in missing_ranges.items():
        if not isinstance(ranges, (list, tuple)):
            ranges = [ranges]

        for r in ranges:
            if isinstance(r, dict):
                low = r.get("lo")
                high = r.get("hi")
            elif isinstance(r, (list, tuple)) and len(r) >= 2:
                low, high = r[0], r[1]
            else:
                low, high = None, None

            missing_rows.append({
                "chapter": chapter,
                "source_file": source_file,
                "target_table": target_table,
                "variable_name": var_name,
                "missing_type": "range",
                "missing_value": str(r),
                "missing_range_low": str(low) if low is not None else None,
                "missing_range_high": str(high) if high is not None else None,
                "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            })

# --------------------------------------------------------
# Convert to DataFrames
# --------------------------------------------------------
variables_df = pd.DataFrame(variables_rows)

value_labels_df = pd.DataFrame(value_label_rows)

missing_df = pd.DataFrame(
    missing_rows,
    columns=[
        "chapter",
        "source_file",
        "target_table",
        "variable_name",
        "missing_type",
        "missing_value",
        "missing_range_low",
        "missing_range_high",
        "checked_at_utc",
    ],
)


# --------------------------------------------------------
# Summary
# --------------------------------------------------------
print("variables:", len(variables_df))
print("value labels:", len(value_labels_df))
print("missing codes:", len(missing_df))

display(variables_df.head())
display(value_labels_df.head())
display(missing_df.head())

In [ ]:
# ============================================================
# 3. Register source files and PDF references
# Corrected for Issue #18: classify questionnaire/dictionary PDFs,
# calculate SHA-256, attempt Drive ID resolution, and create an auditable check.
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def norm_text(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()


def classify_pdf_role(pdf_path):
    """Best-effort classification. Manual registry can override this."""
    name = norm_text(Path(pdf_path).name)

    dictionary_terms = [
        "diccionario", "dictionary", "variables", "variable", "metadata",
        "metadatos", "codigos", "labels", "valores",
    ]
    questionnaire_terms = [
        "cuestionario", "questionnaire", "encuesta", "instrumento",
        "modulo", "modulo", "ficha", "formulario",
    ]

    if any(term in name for term in dictionary_terms):
        return "variable_dictionary"
    if any(term in name for term in questionnaire_terms):
        return "questionnaire"
    return "unclassified"


def find_drive_file_ids_by_name(filename):
    """Resolve Drive IDs by exact filename. May return multiple IDs if duplicated."""
    safe_name = filename.replace("'", "\\'")
    query = f"name = '{safe_name}' and trashed = false"
    try:
        response = drive_service.files().list(
            q=query,
            fields="files(id, name, mimeType)",
            pageSize=10,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        return [item["id"] for item in response.get("files", [])]
    except Exception as exc:
        print(f"Warning: Drive ID lookup failed for {filename}: {exc}")
        return []


manual_registry_path = f"{LOG_DIR}/ENARES_2024_STAGE2_pdf_registry_manual.csv"
manual_columns = [
    "module", "chapter", "sav_file", "target_table",
    "questionnaire_pdf_file", "questionnaire_pdf_path", "questionnaire_pdf_drive_id",
    "variable_dictionary_pdf_file", "variable_dictionary_pdf_path", "variable_dictionary_pdf_drive_id",
    "notes",
]

if os.path.exists(manual_registry_path):
    manual_registry = pd.read_csv(manual_registry_path)
    for col in manual_columns:
        if col not in manual_registry.columns:
            manual_registry[col] = None
else:
    manual_registry = pd.DataFrame(columns=manual_columns)
    manual_registry.to_csv(manual_registry_path, index=False)
    print("Manual PDF registry template created:", manual_registry_path)
    print("Fill it if automatic PDF classification or Drive ID resolution is incomplete.")


def get_manual_value(row, field):
    if manual_registry.empty:
        return None
    mask = (
        (manual_registry["module"].astype(str) == str(row["module"])) &
        (manual_registry["chapter"].astype(str) == str(row["chapter"])) &
        (manual_registry["sav_file"].astype(str) == str(row["source_file"])) &
        (manual_registry["target_table"].astype(str) == str(row["target_table"]))
    )
    matches = manual_registry.loc[mask]
    if matches.empty or field not in matches.columns:
        return None
    value = matches.iloc[0][field]
    if pd.isna(value) or str(value).strip() == "":
        return None
    return str(value).strip()


def pick_first_pdf(pdfs, role):
    candidates = [p for p in pdfs if classify_pdf_role(p) == role]
    return candidates[0] if candidates else None


source_rows = []

for _, row in table_mapping.iterrows():
    sav_path = Path(row["source_path"])

    # Look for PDFs near the .sav file and its parent folders.
    module_root = sav_path.parents[2] if len(sav_path.parents) >= 3 else sav_path.parent
    pdfs = sorted([str(p) for p in module_root.rglob("*.pdf")])

    pdf_names = [Path(p).name for p in pdfs]
    pdf_hashes = [sha256_file(p) for p in pdfs]
    pdf_roles = [classify_pdf_role(p) for p in pdfs]

    # Manual registry overrides automatic classification.
    q_path = get_manual_value(row, "questionnaire_pdf_path") or pick_first_pdf(pdfs, "questionnaire")
    d_path = get_manual_value(row, "variable_dictionary_pdf_path") or pick_first_pdf(pdfs, "variable_dictionary")

    q_file = get_manual_value(row, "questionnaire_pdf_file") or (Path(q_path).name if q_path else None)
    d_file = get_manual_value(row, "variable_dictionary_pdf_file") or (Path(d_path).name if d_path else None)

    q_sha = sha256_file(q_path) if q_path and os.path.exists(q_path) else None
    d_sha = sha256_file(d_path) if d_path and os.path.exists(d_path) else None

    q_drive_id = get_manual_value(row, "questionnaire_pdf_drive_id")
    d_drive_id = get_manual_value(row, "variable_dictionary_pdf_drive_id")

    # Try Drive API resolution by exact filename if manual ID is missing.
    q_drive_candidates = find_drive_file_ids_by_name(q_file) if q_file and not q_drive_id else []
    d_drive_candidates = find_drive_file_ids_by_name(d_file) if d_file and not d_drive_id else []

    if not q_drive_id and len(q_drive_candidates) == 1:
        q_drive_id = q_drive_candidates[0]
    if not d_drive_id and len(d_drive_candidates) == 1:
        d_drive_id = d_drive_candidates[0]

    q_status = "complete" if q_file and q_sha and q_drive_id else "incomplete"
    d_status = "complete" if d_file and d_sha and d_drive_id else "incomplete"

    source_rows.append({
        "module": row["module"],
        "chapter": row["chapter"],
        "sav_file": row["source_file"],
        "sav_path": row["source_path"],
        "sav_sha256": sha256_file(row["source_path"]),
        "target_table": row["target_table"],
        "pdf_files_found": "; ".join(pdf_names),
        "pdf_paths_found": "; ".join(pdfs),
        "pdf_sha256_found": "; ".join(pdf_hashes),
        "pdf_roles_detected": "; ".join([f"{Path(p).name}:{role}" for p, role in zip(pdfs, pdf_roles)]),
        "questionnaire_pdf_file": q_file,
        "questionnaire_pdf_path": q_path,
        "questionnaire_pdf_sha256": q_sha,
        "questionnaire_pdf_drive_id": q_drive_id,
        "questionnaire_pdf_drive_id_candidates": "; ".join(q_drive_candidates),
        "questionnaire_pdf_status": q_status,
        "variable_dictionary_pdf_file": d_file,
        "variable_dictionary_pdf_path": d_path,
        "variable_dictionary_pdf_sha256": d_sha,
        "variable_dictionary_pdf_drive_id": d_drive_id,
        "variable_dictionary_pdf_drive_id_candidates": "; ".join(d_drive_candidates),
        "variable_dictionary_pdf_status": d_status,
        "pdf_dictionary_extracted_to_table": "no",
        "pdf_dictionary_extraction_notes": (
            "Official PDF metadata is complete when questionnaire/dictionary file, SHA-256, "
            "and Drive ID are present. If Drive ID candidates are ambiguous or missing, fill "
            "ENARES_2024_STAGE2_pdf_registry_manual.csv and rerun this notebook."
        ),
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    })

source_files_df = pd.DataFrame(source_rows)

source_files_df["pdfs_found"] = source_files_df["pdf_files_found"].fillna("").str.len() > 0
source_files_df["pdf_governance_complete"] = (
    (source_files_df["questionnaire_pdf_status"] == "complete") &
    (source_files_df["variable_dictionary_pdf_status"] == "complete")
)
source_files_df["pdf_preservation_status"] = source_files_df["pdf_governance_complete"].map({
    True: "official_questionnaire_and_dictionary_registered_with_sha256_and_drive_id",
    False: "incomplete_official_pdf_governance_metadata",
})
source_files_df["pdf_limitation_note"] = source_files_df.apply(
    lambda r: (
        "Complete: questionnaire and dictionary PDFs have file name, SHA-256 and Drive ID."
        if r["pdf_governance_complete"]
        else "Incomplete: fill/verify ENARES_2024_STAGE2_pdf_registry_manual.csv for official PDF classification and Drive IDs."
    ),
    axis=1,
)

display(source_files_df)

source_files_output = f"{LOG_DIR}/ENARES_2024_STAGE2_metadata_source_files.csv"
source_files_df.to_csv(source_files_output, index=False)

print(f"Source files metadata saved to: {source_files_output}")
print(f"Manual PDF registry path: {manual_registry_path}")

In [ ]:
# ============================================================
# 4. Load metadata tables to BigQuery
# ============================================================

dataset_id = "enares2024_crs04_raw"

variables_df.to_gbq(
    f"{dataset_id}.metadata_crs04_variables",
    project_id=PROJECT_ID,
    if_exists="replace"
)

value_labels_df.to_gbq(
    f"{dataset_id}.metadata_crs04_value_labels",
    project_id=PROJECT_ID,
    if_exists="replace"
)

source_files_df.to_gbq(
    f"{dataset_id}.metadata_crs04_source_files",
    project_id=PROJECT_ID,
    if_exists="replace"
)

print("Loaded variables, value labels, and source files metadata.")

In [ ]:
# ============================================================
# 4.1 Issue #18 competency check: PDF metadata and governance
# Required output: ENARES_2024_STAGE2_competency_pdf_metadata_check.csv
# ============================================================

required_pdf_columns = [
    "questionnaire_pdf_file",
    "questionnaire_pdf_sha256",
    "questionnaire_pdf_drive_id",
    "variable_dictionary_pdf_file",
    "variable_dictionary_pdf_sha256",
    "variable_dictionary_pdf_drive_id",
    "pdf_dictionary_extracted_to_table",
    "pdf_dictionary_extraction_notes",
]

source_schema = {field.name for field in client.get_table(f"{PROJECT_ID}.{METADATA_DATASET}.metadata_crs04_source_files").schema}

schema_checks = [
    {
        "check_type": "schema_column_present",
        "module": None,
        "chapter": None,
        "target_table": "metadata_crs04_source_files",
        "required_item": col,
        "present": col in source_schema,
        "pass": col in source_schema,
        "evidence": f"{PROJECT_ID}.{METADATA_DATASET}.metadata_crs04_source_files",
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    for col in required_pdf_columns
]

row_checks = []
for _, r in source_files_df.iterrows():
    row_checks.extend([
        {
            "check_type": "questionnaire_pdf_complete",
            "module": r["module"],
            "chapter": r["chapter"],
            "target_table": r["target_table"],
            "required_item": "questionnaire_pdf_file + questionnaire_pdf_sha256 + questionnaire_pdf_drive_id",
            "present": bool(r.get("questionnaire_pdf_file")) and bool(r.get("questionnaire_pdf_sha256")) and bool(r.get("questionnaire_pdf_drive_id")),
            "pass": bool(r.get("questionnaire_pdf_file")) and bool(r.get("questionnaire_pdf_sha256")) and bool(r.get("questionnaire_pdf_drive_id")),
            "evidence": source_files_output,
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        {
            "check_type": "variable_dictionary_pdf_complete",
            "module": r["module"],
            "chapter": r["chapter"],
            "target_table": r["target_table"],
            "required_item": "variable_dictionary_pdf_file + variable_dictionary_pdf_sha256 + variable_dictionary_pdf_drive_id",
            "present": bool(r.get("variable_dictionary_pdf_file")) and bool(r.get("variable_dictionary_pdf_sha256")) and bool(r.get("variable_dictionary_pdf_drive_id")),
            "pass": bool(r.get("variable_dictionary_pdf_file")) and bool(r.get("variable_dictionary_pdf_sha256")) and bool(r.get("variable_dictionary_pdf_drive_id")),
            "evidence": source_files_output,
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        },
    ])

pdf_metadata_check = pd.DataFrame(schema_checks + row_checks)
pdf_metadata_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_competency_pdf_metadata_check.csv"
pdf_metadata_check.to_csv(pdf_metadata_check_output, index=False)

display(pdf_metadata_check)
print("PDF metadata competency check saved:", pdf_metadata_check_output)

# Do not fail the whole Notebook 03 automatically: Notebook 03 can preserve SPSS metadata
# even when official PDF Drive IDs still need manual/API registration. The integrated
# closure notebook should require all pass == True for Issue #18.
pdf_metadata_governance_pass = bool(pdf_metadata_check["pass"].all())
print("pdf_metadata_governance_pass:", pdf_metadata_governance_pass)

if not pdf_metadata_governance_pass:
    incomplete = pdf_metadata_check.loc[~pdf_metadata_check["pass"], ["check_type", "module", "chapter", "target_table", "required_item"]]
    print("Incomplete PDF governance items. Fill manual registry and rerun if required for final closure:")
    display(incomplete)

In [ ]:
# ============================================================
# 5. Load missing codes table
# If missing_df is empty, create a physical empty table with schema.
# ============================================================

missing_table_id = (
    f"{PROJECT_ID}.{METADATA_DATASET}.metadata_crs04_missing_codes"
)

missing_schema = [
    bigquery.SchemaField("chapter", "STRING"),
    bigquery.SchemaField("source_file", "STRING"),
    bigquery.SchemaField("target_table", "STRING"),
    bigquery.SchemaField("variable_name", "STRING"),
    bigquery.SchemaField("missing_type", "STRING"),
    bigquery.SchemaField("missing_value", "STRING"),
    bigquery.SchemaField("missing_range_low", "STRING"),
    bigquery.SchemaField("missing_range_high", "STRING"),
    bigquery.SchemaField("checked_at_utc", "STRING"),
]

if len(missing_df) > 0:
    missing_df.to_gbq(
        f"{METADATA_DATASET}.metadata_crs04_missing_codes",
        project_id=PROJECT_ID,
        if_exists="replace",
    )

    print("Loaded missing codes metadata.")

else:
    table = bigquery.Table(
        missing_table_id,
        schema=missing_schema,
    )

    client.delete_table(
        missing_table_id,
        not_found_ok=True,
    )

    client.create_table(table)

    print(
        "No missing codes detected. Created empty physical "
        "metadata_crs04_missing_codes table with explicit schema."
    )

In [ ]:
# ============================================================
# 6. Create metadata inventory
# Required output: ENARES_2024_STAGE2_metadata_inventory.csv
# ============================================================

metadata_inventory = []

for table_name in METADATA_TABLES:
    full_table_id = (
        f"{PROJECT_ID}.{METADATA_DATASET}.{table_name}"
    )

    try:
        table = client.get_table(full_table_id)
        table_exists = True
        error_message = None

    except Exception as e:
        table = None
        table_exists = False
        error_message = str(e)

    metadata_inventory.append({
        "project_id": PROJECT_ID,
        "dataset_id": METADATA_DATASET,
        "table_name": table_name,
        "full_table_id": full_table_id,
        "table_exists": table_exists,
        "row_count": table.num_rows if table_exists else None,
        "column_count": len(table.schema) if table_exists else None,
        "error_message": error_message,
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    })

metadata_inventory = pd.DataFrame(metadata_inventory)

metadata_inventory_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_metadata_inventory.csv"
)

metadata_inventory.to_csv(
    metadata_inventory_output,
    index=False,
)

display(metadata_inventory)

if not metadata_inventory["table_exists"].all():
    raise RuntimeError(
        "One or more required metadata tables do not exist in BigQuery."
    )

if not os.path.exists(metadata_inventory_output):
    raise FileNotFoundError(
        "Required metadata inventory CSV was not created."
    )

print(
    f"Metadata inventory saved to: {metadata_inventory_output}"
)

In [ ]:
# ============================================================
# 7. Final acceptance check
# ============================================================

expected_outputs = [
    "ENARES_2024_STAGE2_metadata_inventory.csv",
    "ENARES_2024_STAGE2_metadata_source_files.csv",
    "ENARES_2024_STAGE2_competency_pdf_metadata_check.csv",
    "ENARES_2024_STAGE2_pdf_registry_manual.csv",
]

missing_outputs = [
    filename
    for filename in expected_outputs
    if not os.path.exists(f"{LOG_DIR}/{filename}")
]

if missing_outputs:
    raise FileNotFoundError(
        f"Missing required metadata outputs: {missing_outputs}"
    )

if not metadata_inventory["table_exists"].all():
    raise RuntimeError(
        "Not all required metadata tables were verified in BigQuery."
    )

print("\nNotebook 3 completed successfully.")

print("Required outputs verified:")
for filename in expected_outputs:
    print(f" - {filename}")

print("\nStage 2 SPSS metadata preserved.")
print("Issue #18 PDF governance check generated:", pdf_metadata_check_output)
print("pdf_metadata_governance_pass:", pdf_metadata_governance_pass)

print(
    "No merge, recoding, indicators, "
    "or statistical analysis were performed."
)

# Supplementary checks

In [ ]:
# ============================================================
# Supplementary controls
# These controls are useful audit artifacts, but they are not
# mandatory for closing Notebook 03.
# ============================================================

# ============================================================
# 8. Control C3 variables inside CRS04
# ============================================================
c3_sql = f"""
SELECT
    chapter,
    source_file,
    variable_name,
    variable_label
FROM `{PROJECT_ID}.{METADATA_DATASET}.metadata_crs04_variables`
WHERE STARTS_WITH(variable_name, "C3")
ORDER BY chapter, variable_name
"""

c3_variables = (
    client.query(c3_sql)
    .result()
    .to_dataframe()
)

c3_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_c3_variables_in_crs04_check.csv"
)

c3_variables.to_csv(
    c3_output,
    index=False,
)

print(f"Saved: {c3_output}")
display(c3_variables.head(50))

# ============================================================
# 9. Prepare key-validation SQL for Stage 3 only
# Do not execute merge in Stage 2
# ============================================================
key_preview_sql = f"""
-- CONSULTA PARA FASE 03. NO CREA TABLAS CLEANED EN STAGE 2.

SELECT
    "CAP100" AS tabla,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID)) AS distinct_key
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100`

UNION ALL

SELECT
    "CAP200",
    COUNT(*),
    COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200`

UNION ALL

SELECT
    "CAP248",
    COUNT(*),
    COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap248`

UNION ALL

SELECT
    "CAP300",
    COUNT(*),
    COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap300`
"""

key_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_key_validation_preview_for_stage3.sql"
)

with open(key_output, "w", encoding="utf-8") as f:
    f.write(key_preview_sql)

print(f"Saved: {key_output}")
print(key_preview_sql)

# ============================================================
# 10. Save FLOAT64 key warning SQL for Stage 3
# ============================================================
float_key_check_sql = f"""
-- USAR EN FASE 03 SI ID O COLEGIAL_ID LLEGAN COMO FLOAT64.
-- Esta consulta NO convierte llaves. Solo prepara la validacion.

SELECT
    "raw_crs04_cap100" AS tabla,
    COUNTIF(ID IS NULL) AS id_nulls,
    COUNTIF(COLEGIAL_ID IS NULL) AS colegial_id_nulls,
    COUNTIF(ID IS NOT NULL AND ID != FLOOR(ID)) AS id_with_decimals,
    COUNTIF(COLEGIAL_ID IS NOT NULL AND COLEGIAL_ID != FLOOR(COLEGIAL_ID))
        AS colegial_id_with_decimals
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100`
"""

float_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_float64_key_warning_for_stage3.sql"
)

with open(float_output, "w", encoding="utf-8") as f:
    f.write(float_key_check_sql)

print(f"Saved: {float_output}")
print(float_key_check_sql)

# ============================================================
# 11. Document preservation of complex survey design variables
# ============================================================
design_vars = [
    "CCDD",
    "ID",
    "ID_AULA",
]

design_sql = f"""
SELECT
    chapter,
    source_file,
    variable_name,
    variable_label
FROM `{PROJECT_ID}.{METADATA_DATASET}.metadata_crs04_variables`
WHERE variable_name IN UNNEST(@design_vars)
ORDER BY variable_name, chapter
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "design_vars",
            "STRING",
            design_vars,
        )
    ]
)

design_var_check = (
    client.query(
        design_sql,
        job_config=job_config,
    )
    .result()
    .to_dataframe()
)

design_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_design_variables_preservation_check.csv"
)

design_var_check.to_csv(
    design_output,
    index=False,
)

print(f"Saved: {design_output}")
display(design_var_check)

## Issue 24

In [ ]:
raw_tables = [
    'raw_crs04_cap100',
    'raw_crs04_cap200',
    'raw_crs04_cap248',
    'raw_crs04_cap300'
]

validation_sql = []

for table in raw_tables:
    validation_sql.append(f"""
    SELECT
      '{table}' AS table_name,
      COUNT(*) AS total_rows,
      COUNTIF(ID IS NULL) AS id_nulls,
      COUNTIF(COLEGIAL_ID IS NULL) AS colegial_id_nulls,
      COUNT(DISTINCT CONCAT(CAST(ID AS STRING), '||', CAST(COLEGIAL_ID AS STRING))) AS distinct_keys,
      COUNT(*) - COUNT(DISTINCT CONCAT(CAST(ID AS STRING), '||', CAST(COLEGIAL_ID AS STRING))) AS duplicated_key_rows
    FROM `{PROJECT_ID}.enares2024_crs04_raw.{table}`
    """)

key_check = client.query(
    " UNION ALL ".join(validation_sql)
).result().to_dataframe()

display(key_check)

## Issue  25

In [ ]:
schema_rows = []

for table in [
    'raw_crs04_cap100',
    'raw_crs04_cap200',
    'raw_crs04_cap248',
    'raw_crs04_cap300'
]:
    t = client.get_table(f'{PROJECT_ID}.enares2024_crs04_raw.{table}')
    for field in t.schema:
        if field.name in ['ID', 'COLEGIAL_ID']:
            schema_rows.append({
                'table': table,
                'column': field.name,
                'type': field.field_type
            })

pd.DataFrame(schema_rows)

In [ ]:
float_checks = []

for table in [
    'raw_crs04_cap100',
    'raw_crs04_cap200',
    'raw_crs04_cap248',
    'raw_crs04_cap300'
]:
    q = f"""
    SELECT
      '{table}' AS table_name,
      COUNTIF(ID IS NOT NULL AND ID != FLOOR(ID)) AS id_with_decimals,
      COUNTIF(COLEGIAL_ID IS NOT NULL AND COLEGIAL_ID != FLOOR(COLEGIAL_ID)) AS colegial_id_with_decimals
    FROM `{PROJECT_ID}.enares2024_crs04_raw.{table}`
    """
    float_checks.append(client.query(q).result().to_dataframe())

float_check = pd.concat(float_checks, ignore_index=True)
display(float_check)

In [ ]:
key_check.to_csv(f'{LOG_DIR}/stage3_key_validation.csv', index=False)
float_check.to_csv(f'{LOG_DIR}/stage3_float_key_check.csv', index=False)